In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import *
from statsmodels.stats import weightstats, proportion

Определим мощность теста:  
beta = 0.2 (стандарт) - вероятность ошибки II рода
Power = 1-beta = 0.8 (80%)

Зафиксируем минимальный эффект, который мы ожидаем от теста:  
MDE = 5% (должен определяться до начала тестирования)  

Уровень значимости:   
alpha = 0.05  (стандарт) - вероятность ошибки I рода. (должен определяться до запуска тестирования).  

Ключевая метрика для расчета размера выборки:  
СR_to_paid (или Purchases_per_user). Проведем расчеты ниже

In [3]:
# берем данные
data = pd.read_csv('../data/data_sessions.csv', parse_dates=['datetime', 'date'])

# объединение контрольных групп
data['Group'] = data['ExpId'].map({246: 'A', 247: 'A', 248: 'B'})

In [4]:
# проверка 
data.groupby('ExpId',group_keys=False).apply(lambda x: x.sample(2))

C:\Users\User\AppData\Local\Temp\ipykernel_6440\1754687543.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data.groupby('ExpId',group_keys=False).apply(lambda x: x.sample(2))


,EventName,DeviceIDHash,ExpId,datetime,date,session,Group
84885,CartScreenAppear,3337471580007169353,246,2019-08-07 11:36:15,2019-08-07,23,A
196113,MainScreenAppear,7450040783378882729,246,2019-08-03 06:13:56,2019-08-03,2,A
159991,CartScreenAppear,6154340184587822225,247,2019-08-05 13:59:54,2019-08-05,7,A
129015,MainScreenAppear,4890328091489891497,247,2019-08-06 19:34:59,2019-08-06,11,A
87042,MainScreenAppear,3385440971612364263,248,2019-08-02 15:32:53,2019-08-02,6,B
196595,MainScreenAppear,7483596365340385091,248,2019-08-02 08:18:07,2019-08-02,3,B


In [6]:
data['users_in_group'] = data.groupby('Group')['DeviceIDHash'].transform('nunique')

# Расчет метрик 
events_uniq_user_in_group = (data
                        .groupby(['Group', 'EventName'])
                        .agg(uniq_user = ('DeviceIDHash', 'nunique'),
                             users_in_group=('users_in_group', 'first'))
                        .reset_index()
)
#event_name = list(events_uniq_user_in_group['EventName'].unique()) # список названий событий
events_uniq_user_in_group['CR'] = events_uniq_user_in_group['uniq_user'] / events_uniq_user_in_group['users_in_group']

#events_uniq_user_in_group.query("EventName == 'PaymentScreenSuccessful' ")  #'[events_uniq_user_in_group['EventName']]


In [7]:
events_uniq_user_in_group 

,Group,EventName,uniq_user,users_in_group,CR
0,A,CartScreenAppear,2504,4997,0.501101
1,A,MainScreenAppear,4926,4997,0.985791
2,A,OffersScreenAppear,3062,4997,0.612768
3,A,PaymentScreenSuccessful,2358,4997,0.471883
4,A,Tutorial,561,4997,0.112267
5,B,CartScreenAppear,1230,2537,0.484825
6,B,MainScreenAppear,2493,2537,0.982657
7,B,OffersScreenAppear,1531,2537,0.603469
8,B,PaymentScreenSuccessful,1181,2537,0.465510
9,B,Tutorial,279,2537,0.109972


ТЕСТЫ. КАКАЯ ЖЕ МЕТРИКА ПОДОЙДЕТ, ЧТОБЫ СЧИТАТЬ РАЗМЕР ВЫБОРКИ ПРИЕМЛЕМЫМ.


In [8]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import tt_ind_solve_power, NormalIndPower
import math

In [9]:
# CR_paid группы А
CR_paid_A = events_uniq_user_in_group.query("(EventName == 'PaymentScreenSuccessful') & (Group == 'A') ")['CR']
CR_paid_A_actual = CR_paid_A.iloc[0]
MDE = 0.05
CR_paid_B_planned = CR_paid_A_actual * (1 + MDE)
alpha = 0.05
power = 0.8

effect_size = proportion_effectsize(prop1=CR_paid_B_planned, prop2=CR_paid_A_actual)
sample_size_per_group = tt_ind_solve_power(effect_size=effect_size,
                                           alpha=alpha,
                                           power=power,
                                           nobs1=None,        # Указываем None, чтобы функция рассчитала nobs1
                                           ratio=1.0,         # Соотношение групп 1:1
                                           alternative='two-sided' # Двусторонний тест 
)
print(f"Необходимый размер выборки на одну группу: {np.ceil(sample_size_per_group):.0f} пользователей")
print(f"Общий размер выборки (обе группы): {np.ceil(sample_size_per_group * 2):.0f} пользователей")

# продолжительность теста
first_day = data.groupby(['Group', 'DeviceIDHash'])['date'].min().reset_index()
# Сколько новых пользователей в день в каждой группе
new_by_day = first_day.groupby(['Group', 'date']).size().reset_index(name='new_users')
# Среднее для группы B (так как она меньше)
avg_new_B = new_by_day[new_by_day['Group'] == 'B']['new_users'].mean()

days_needed = sample_size_per_group / avg_new_B
print(f'Продолжительность теста с текущим средним приростом уникальных пользователей в день: {int(days_needed) + 1}')

#Необходимый размер выборки на одну группу: 7042 пользователей. Продолжительность теста не менее 20 дней. Но у нас лог за 7 дней.
# а мы имеем в группе А 4997, в группе В 2537, вместо 7042 

# либо изменяем параметры теста (эффект подклядывания - плохо), либо исходим из факта "что у нас есть" и пересчитываем мощность теста при MDE = 5% и решаем что делать

# еще вариант - выбираем целевой метрикой другую (например, Purchases_per_session, Purchases_per_user)   - вариант получился провальным, с данными метриками можности были 
# значительно хуже, чем для CR_paid




Необходимый размер выборки на одну группу: 7042 пользователей
Общий размер выборки (обе группы): 14084 пользователей
Продолжительность теста с текущим средним приростом уникальных пользователей в день: 20


In [10]:
# попробуем расчитать реальную мощность теста 

# размер выборок после теста
n_A_actual = 4997   # уникальных пользователей в группе A
n_B_actual = 2537   # уникальных пользователей в группе B
ratio = n_A_actual / n_B_actual  # 2537 / 4997 ≈ 0.5077 - реальное соотношение групп

# Вариант 1 Рассчитываем мощность для бизнес-значимого эффекта 
CR_paid_B_planned = CR_paid_A_actual * (1 + MDE)
effect_size_actual = proportion_effectsize(CR_paid_B_planned, CR_paid_A_actual)
# Расчет мощности 
power_MDE = tt_ind_solve_power(effect_size=effect_size_actual,
                                  nobs1=n_B_actual,      # размер меньшей группы (лимитирующей)
                                  alpha=0.05,
                                  ratio=ratio,
                                  power=None,
                                  alternative='two-sided'
)
print(f'Для MDE=5% и текущей выборкой значение мощности составляет {power_MDE}')

# Вариант 2 Расчитываем можность для фактического показателя метрики CR_paid
# для этого нужно найти актуальную CR_paid_B
CR_paid_B = events_uniq_user_in_group.query("(EventName == 'PaymentScreenSuccessful') & (Group == 'B') ")['CR']
CR_paid_B_actual = CR_paid_B.iloc[0]
effect_size_actual = proportion_effectsize(CR_paid_B_actual, CR_paid_A_actual)
# Расчет фактической мощности
power_actual = tt_ind_solve_power(effect_size=effect_size_actual,
                                  nobs1=n_B_actual,      # размер меньшей группы (лимитирующей)
                                  alpha=0.05,
                                  ratio=ratio,
                                  power=None,
                                  alternative='two-sided'
)
print(f'Учитывая текущую конверсию в группе В, и зафиксированное количество пользователей, имеем текущую мощность теста: {power_actual}')

Для MDE=5% и текущей выборкой значение мощности составляет 0.49075217812183464
Учитывая текущую конверсию в группе В, и зафиксированное количество пользователей, имеем текущую мощность теста: 0.08197635753848352


In [11]:
n_A_actual = 4997   # уникальных пользователей в группе A
n_B_actual = 2537   # уникальных пользователей в группе B
ratio = n_A_actual / n_B_actual  # 2537 / 4997 ≈ 0.5077 - реальное соотношение групп
mde_h = tt_ind_solve_power(effect_size=None,
                                  nobs1=n_B_actual,      # размер меньшей группы (лимитирующей). у нас это тест
                                  alpha=0.05,
                                  ratio=ratio,   # контроль / тест
                                  power=0.8,
                                  alternative='two-sided'
)
# переведем mde_h в абсолютный MDE
def h_to_abs_mde(h, p_control):
    """Перевод Cohen's h в абсолютный MDE"""
    # Обратная формула: p_test = [sin(h/2 + arcsin(√p_control))]²
    p_test = (math.sin(h/2 + math.asin(math.sqrt(p_control)))) ** 2
    return p_test - p_control

# Использование
p_control = 0.47188
h = 0.0683

mde_abs = h_to_abs_mde(mde_h, CR_paid_A_actual)
print(f"Абсолютный MDE: {mde_abs:.4f} ({mde_abs*100:.2f} п.п.)")
print(f"Относительный MDE: {mde_abs/p_control:.1%}")

Абсолютный MDE: 0.0341 (3.41 п.п.)
Относительный MDE: 7.2%


In [12]:
power_calculator = NormalIndPower()
power = power_calculator.solve_power(
    effect_size=effect_size,
    nobs1=n_B_actual,
    alpha=0.05,
    ratio=ratio,
    alternative='two-sided'
)
power

np.float64(0.4908506275920568)

In [14]:
# Сравним CR_paid_A и CR_paid_B 
CR_paid_AB = events_uniq_user_in_group.query("EventName == 'PaymentScreenSuccessful' ")
observed = CR_paid_AB['uniq_user'].to_list()
nobs = CR_paid_AB['users_in_group'].to_list()
z_stat, p = proportion.proportions_ztest(observed, nobs)
print(f'Значение pv-value = {p} > 0.05. Статистически значимых различий не обнаружено')


0.6004294282308703


In [18]:
## вычиляем pvalue по каждой метрике и добавляем в словарь
pvalue_dict = {}
events_uniq_user_in_group = events_uniq_user_in_group.sort_values('Group')
event_name = list(events_uniq_user_in_group['EventName'].unique()) # список названий событий
for event in event_name:
    observed = events_uniq_user_in_group[events_uniq_user_in_group['EventName'] == event]['uniq_user'].to_list()
    nobs = events_uniq_user_in_group[events_uniq_user_in_group['EventName'] == event]['users_in_group'].to_list() 
    z_stat, p = proportion.proportions_ztest(observed, nobs)
    pvalue_dict[event] = (observed, nobs, round(float(p),4))

In [19]:
pvalue_dict

{'CartScreenAppear': ([2504, 1230], [4997, 2537], 0.1818),
 'MainScreenAppear': ([4926, 2493], [4997, 2537], 0.2942),
 'OffersScreenAppear': ([3062, 1531], [4997, 2537], 0.4343),
 'PaymentScreenSuccessful': ([2358, 1181], [4997, 2537], 0.6004),
 'Tutorial': ([561, 279], [4997, 2537], 0.7649)}

In [22]:
active_per_user_in_group = (data
                            .groupby(['Group', 'DeviceIDHash'])
                            .agg(Offers_per_user = ('EventName',lambda x: (x == 'OffersScreenAppear').sum()),
                                Purchases_per_user = ('EventName',lambda x: (x == 'PaymentScreenSuccessful').sum()),
                                Sessions_per_user = ('session', 'nunique')
                                )
).reset_index()
# добавляем столбец Purchases_per_session
active_per_user_in_group['Purchases_per_session'] = active_per_user_in_group['Purchases_per_user'] / active_per_user_in_group['Sessions_per_user']
metrics = list(active_per_user_in_group.columns)[2:] # список названий метрик (названий столбцов)
## вычиляем pvalue по каждой метрике и добавляем в словарь
for metric in metrics:
    value_A = active_per_user_in_group[active_per_user_in_group['Group'] == 'A'][metric]
    value_B = active_per_user_in_group[active_per_user_in_group['Group'] == 'B'][metric]
    t_stat, t_p = ttest_ind(value_A, value_B, equal_var=False)
    pvalue_dict[metric] = round(float(t_p),4) # результаты добавляем в словарь


In [25]:
pvalue_dict

{'CartScreenAppear': ([2504, 1230], [4997, 2537], 0.1818),
 'MainScreenAppear': ([4926, 2493], [4997, 2537], 0.2942),
 'OffersScreenAppear': ([3062, 1531], [4997, 2537], 0.4343),
 'PaymentScreenSuccessful': ([2358, 1181], [4997, 2537], 0.6004),
 'Tutorial': ([561, 279], [4997, 2537], 0.7649),
 'Offers_per_user': 0.193,
 'Purchases_per_user': 0.5469,
 'Sessions_per_user': 0.939,
 'Purchases_per_session': 0.7178}

In [27]:
active_per_user_in_group.groupby('Group').mean()

,DeviceIDHash,Offers_per_user,Purchases_per_user,Sessions_per_user,Purchases_per_session
Group,,,,,
A,4.680018e+18,5.994597,4.393236,5.983590,0.570981
B,4.667490e+18,6.462357,4.793063,5.994088,0.587930
